In [ ]:
import os
import boto3
from config import *
import base64

In [ ]:
print(boto3.__version__)

In [ ]:
bedrock_agent = boto3.Session().client('bedrock-agent')

In [ ]:
path = "/home/sagemaker-user/.cache/kagglehub/datasets/hsankesara/flickr-image-dataset/versions/1/flickr30k_images/flickr30k_images"
print(path)

In [ ]:
counter = 0
for filename in os.listdir(path):
    if counter %1000 == 0:
        print(f"{counter}th == {filename}")
    counter +=1

In [ ]:
# If you're using PIL/Pillow Image object

In [ ]:
## updated this section
def image_to_bytes(image_path):
    with open(image_path, "rb") as image_file:
        # Read the file content
        binary_data = image_file.read()
        # Encode the binary data to base64
        base64_encoded = base64.b64encode(binary_data)
        # Convert bytes to string
        base64_string = base64_encoded.decode('utf-8')     #  <---- adding this step
    return {'data': base64_string}

In [ ]:
## example usage
image_path = path + '/1314231418.jpg'
image_bytes = image_to_bytes(image_path)
image_bytes_data = image_bytes['data']

In [ ]:
print(type(image_bytes['data']))

In [ ]:
import random
import string
import time

In [ ]:
def generate_random_code():
    # Generate 4 random characters (uppercase letters)
    characters = ''.join(random.choices(string.ascii_uppercase, k=4))

    # Generate 12 digit number combining timestamp and random numbers
    timestamp = str(int(time.time()))
    random_digits = ''.join(random.choices(string.digits, k=6))
    number = (timestamp + random_digits)[-12:]  # Ensure 12 digits

    return characters, number

In [ ]:
# now we will test uploading a single image
def upload_image(image_path):
    image_bytes = image_to_bytes(image_path)
    image_bytes_data = image_bytes['data']

    random_code = generate_random_code()
    documentIdentifier = str(random_code[0]) + str(random_code[1])
    document = {
        'content': {
            'custom': {
                'customDocumentIdentifier': {
                    'id': documentIdentifier
                },
                'inlineContent': {
                    'byteContent': {
                        'data': image_bytes_data,
                        'mimeType': 'image/jpeg'
                    },
                    'type': 'BYTE'
                },
                'sourceType': 'IN_LINE'
            },
            'dataSourceType': 'CUSTOM',
        }
    }
    documents = [document]

    response = bedrock_agent.ingest_knowledge_base_documents(
        dataSourceId = data_source_id, documents = documents, knowledgeBaseId = knowledgebase_id
    )
    return response

In [ ]:
### inline upload
counter = 0
for filename in os.listdir(path):
    if counter == 3:
        break
    image_path = path + "/" + filename
    upload_response = upload_image(image_path)
    timer = 0
    print(" --- Updload response")
    print(upload_response)
    print(" --- ")
    while timer < 10:
        time.sleep(1)
        timer +=1
    print(image_path)
    counter +=1

I was getting logs of this kind
```
{
    "event_timestamp": 1740509325142,
    "event": {
        "document_location": {
            "customDocument_location": {
                "id": "***"
            },
            "type": "CUSTOM"
        },
        "request_id": "********",
        "data_source_id": "********",
        "status_reasons": [
            "Resource not processed due to a service exception. Please reach out to Amazon Bedrock technical support team for further assistance, or contact technical support at aws.amazon.com/contact-us/."
        ],
        "knowledge_base_arn": "arn:aws:bedrock:us-east-1:**********:knowledge-base/********",
        "status": "RESOURCE_IGNORED"
    },
    "event_version": "1.0",
    "event_type": "IngestKnowledgeBaseDocuments.ResourceStatusChanged",
    "level": "WARN"
}
```

now I am getting this error:

```
{
    "event_timestamp": 1741107340497,
    "event": {
        "document_location": {
            "customDocument_location": {
                "id": "**********"
            },
            "type": "CUSTOM"
        },
        "request_id": "*********",
        "data_source_id": "*******",
        "status_reasons": [
            "The server encountered an internal error while processing the request."
        ],
        "knowledge_base_arn": "arn:aws:bedrock:us-east-1:794038231401:knowledge-base/5WYB0DMRMN",
        "status": "RESOURCE_IGNORED"
    },
    "event_version": "1.0",
    "event_type": "IngestKnowledgeBaseDocuments.ResourceStatusChanged",
    "level": "ERROR"
}
```

uploading 6 GB of data will take an excesive amount of time and resources to test. For instance, when I first attempted to upload a 17 MB pdf file (the bedrock user guide actually) which contains more than 2500 pages, It was taking longer than 8 hours and I had to stop the test for that reason. Hence uploading 6 GB of data will take 118 days.

In [ ]:
s3_location = 's3://mainbucketrockhight5461/test/knowledge-bases/direct_KB_upload/1image/371897.jpg'

In [ ]:
# I will attempt to upload the images using s3 direct upload to custom source
def s3_upload_image(s3_location):

    random_code = generate_random_code()
    documentIdentifier = str(random_code[0]) + str(random_code[1])
    document = {
        'content':{
            'dataSourceType': 'CUSTOM',
            'custom':{
                'customDocumentIdentifier': {
                    'id': documentIdentifier
                },
                's3Location': {
                    'bucketOwnerAccountId': bucket_owner_account,
                    'uri': s3_location
                },
                'sourceType': 'S3_LOCATION'
            }
        }
    }
    print(documentIdentifier)

    documents = [document]

    response = bedrock_agent.ingest_knowledge_base_documents(
        dataSourceId = data_source_id, documents = documents, knowledgeBaseId = knowledgebase_id
    )
    return response

In [ ]:
s3_upload_response = s3_upload_image(s3_location)

In [ ]:
print(s3_upload_response)

In [ ]:
# the request failed, I will test now using a json file
s3_response = s3_upload_image('s3://mainbucketrockhight5461/test/documentation/bedrock-user-guide.pdf.metadata.json')

In [ ]:
print(s3_response)

In [ ]:
## I think the ingestions might be failing because we already reached the limit actually. So I will attempt with a new knowledge base

In [ ]:
# I will attempt to upload the images using s3 direct upload to custom source
def s3_upload_image_nds(s3_location, data_source_id):

    random_code = generate_random_code()
    documentIdentifier = str(random_code[0]) + str(random_code[1])
    document = {
        'content':{
            'dataSourceType': 'CUSTOM',
            'custom':{
                'customDocumentIdentifier': {
                    'id': documentIdentifier
                },
                's3Location': {
                    'bucketOwnerAccountId': bucket_owner_account,
                    'uri': s3_location
                },
                'sourceType': 'S3_LOCATION'
            }
        }
    }

    documents = [document]

    response = bedrock_agent.ingest_knowledge_base_documents(
        dataSourceId = data_source_id, documents = documents, knowledgeBaseId = knowledgebase_id
    )
    return response

# now we will test uploading a single image
def upload_image_nds(image_path, data_source_id):
    image_bytes = pil_image_to_bytes(image_path)
    image_bytes_data = image_bytes['data']

    random_code = generate_random_code()
    documentIdentifier = str(random_code[0]) + str(random_code[1])
    document = {
        'content': {
            'custom': {
                'customDocumentIdentifier': {
                    'id': documentIdentifier
                },
                'inlineContent': {
                    'byteContent': {
                        'data': image_bytes_data,
                        'mimeType': 'image/jpeg'
                    },
                    'type': 'BYTE'
                },
                'sourceType': 'IN_LINE'
            },
            'dataSourceType': 'CUSTOM',
        }
    }
    documents = [document]

    response = bedrock_agent.ingest_knowledge_base_documents(
        dataSourceId = data_source_id, documents = documents, knowledgeBaseId = knowledgebase_id
    )
    return response

In [ ]:
counter = 0
for filename in os.listdir(path):
    if counter == 10:
        break
    image_path = path + "/" + filename
    upload_response = upload_image_nds(image_path, new_data_source)
    print(upload_response)
    timer = 0
    time.sleep(5)
    print(image_path)
    counter +=1

In [ ]:
# uploads are failing manually apparently so I will attempt to upload the image using s3 source

In [ ]:
counter = 0
for filename in os.listdir(path):
    if counter == 10:
        break
    counter +=1
    image_path = path + "/" + filename
    print(image_path)

In [ ]:
s3_response = s3_upload_image_nds('s3://mainbucketrockhight5461/test/knowledge-bases/documentacion.txt', new_data_source)

In [ ]:
# now I will attempt to upload an image
s3_response = s3_upload_image_nds('s3://mainbucketrockhight5461/test/knowledge-bases/direct_KB_upload/1image/371897.jpg', new_data_source)

In [ ]:
print(s3_response)

In [ ]:
# now we will attempt to upload txt files and json files inline

def text_to_bytes(file_path, encoding='utf-8'):
    try:
        with open(file_path, 'r', encoding=encoding) as file:
            content = file.read()
        return content.encode(encoding)
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

def upload_txt_custom(text_path, data_source_id, mime_type):
    bytes_data = text_to_bytes(text_path)

    random_code = generate_random_code()
    documentIdentifier = str(random_code[0]) + str(random_code[1])
    document = {
        'content': {
            'custom': {
                'customDocumentIdentifier': {
                    'id': documentIdentifier
                },
                'inlineContent': {
                    'byteContent': {
                        'data': bytes_data,
                        'mimeType': mime_type
                    },
                    'type': 'BYTE'
                },
                'sourceType': 'IN_LINE'
            },
            'dataSourceType': 'CUSTOM',
        }
    }
    documents = [document]

    response = bedrock_agent.ingest_knowledge_base_documents(
        dataSourceId = data_source_id, documents = documents, knowledgeBaseId = knowledgebase_id
    )
    return response

In [ ]:
mime_type = "text/plain"

In [ ]:
file = "/home/sagemaker-user/user-default-efs/documents/mygithub/aws-samples/bedrock/miscelanious/test.txt"

In [ ]:
file_response = upload_txt_custom(file, new_data_source, mime_type)

In [ ]:
print(file_response)